# TD-trained RNNs on a Pavlovian contingency task

This notebook trains a recurrent network with **temporal-difference (TD) learning** — the learning rule behind the reward-prediction-error theory of dopamine — and reproduces the RNN modeling of

> Qian, Burrell, Hennig, Matias, Murthy, Gershman & Uchida (2025), *[Prospective contingency explains behavior and dopamine signals during associative learning](https://doi.org/10.1038/s41593-025-01915-4)*, Nature Neuroscience.

**Research question.** Midbrain dopamine neurons behave like the TD error of reinforcement learning, $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$. But a TD model is only as good as its state representation: what is "the state" at each moment of an experiment? The paper attacks this question with a Pavlovian *contingency degradation* task in mice. Odor A predicts reward with probability 0.75 and odor B predicts nothing. When extra rewards are then delivered *uncued* during the inter-trial interval (ITI), anticipatory licking and the dopamine response to odor A collapse, even though the odor A → reward contingency itself has not changed. When the same extra rewards are *signaled* by a third odor C, the odor A response survives. Classical contingency measures ($\Delta P$) cannot separate these two conditions; a TD model can, but only if its state representation captures the animal's uncertainty about where it is within the ITI. The paper's best hand-crafted model, the Belief-State model, represents the ITI as a probability distribution over a "Wait" and a "Pre" state. Its final question is the one this notebook is about: instead of hand-crafting states, can a network *learn* an appropriate state representation from observations alone?

**The value-RNN.** A small GRU with a scalar linear readout $V_t$ is trained with TD(0) to predict reward, given only the same cue and reward observations the animal receives. After the same two-phase training as the animals (conditioning first, then degradation or cued reward), the network's TD errors reproduce the dopamine results in all conditions, its pre-cue value explains *why* (uncued rewards make waiting itself valuable), and its hidden units linearly decode the hand-crafted beliefs, the better the larger the network. Trained with the TD error as its only feedback, the RNN discovers a belief-like state space on its own.

## Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from neuralrnn import (
    AutoModel, TDObjective, Trainer, TrainingArguments, load_dataset,
)
from neuralrnn.models.gated_rnn import GatedRNNConfig
from neuralrnn.rl.probe import (
    probe_value_model, split_episode_trials, label_trial_type,
    average_by_aligned_time, mean_rpe_at_events, shift_iti,
)

MODEL_DIR = "models/19"
FIG_DIR = "figs/19"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ---- paper hyperparameters (Qian & Burrell 2024, Methods) ----
GAMMA = 0.83          # TD discount at dt = 0.5 s (0.67 per second)
ISI = 8               # cue -> reward delay in steps (rew_times = [8, 8, 8])
LR = 1e-3             # Adam
BATCH_SIZE = 12       # episodes per batch
TRIALS_PER_EPISODE = 20

# ---- reduced tutorial config ----
N_TRIALS = 2000                  # paper: 10000
STEPS_CONDITIONING = 3000        # paper: 300 epochs ~= 12.5k steps
STEPS_TRANSFER = 2000
SEEDS = [0, 1]                   # paper: 25 simulations
HIDDEN_SWEEP = [5, 10, 20, 50]   # Fig. 6f sweep; 50 units also serve Fig. 6c/d/e/g
HIDDEN_MAIN = 50
MODES = ("conditioning", "degradation", "cue-c")

# shared labels and colors used by all figures
MODEL_FOR_MODE = {"conditioning": "cond", "degradation": "degradation",
                  "cue-c": "cue-c"}          # mode -> checkpoint tag prefix
COND_LABEL = {"conditioning": "Cond", "degradation": "Deg", "cue-c": "CueRew"}
COND_COLOR = {"conditioning": "tab:gray", "degradation": "tab:red",
              "cue-c": "tab:blue"}


def make_dataset(mode, seed, ntrials=N_TRIALS, jitter=1):
    """Training dataset for one mode (reward timing jittered, as the paper)."""
    return load_dataset("contingency", mode=mode, ntrials=ntrials,
                        ntrials_per_episode=TRIALS_PER_EPISODE, jitter=jitter,
                        batch_size=BATCH_SIZE, seed=seed)


def make_test_dataset(mode, seed=1000, ntrials=1000):
    """Jitter-free dataset for display/analysis (paper removes jitter for display)."""
    return make_dataset(mode, seed=seed, ntrials=ntrials, jitter=0)


def new_model(hidden):
    """Value-RNN: GRU + scalar linear readout (PyTorch default init, as the paper)."""
    return AutoModel.from_config(GatedRNNConfig(
        input_dim=4, latent_dim=hidden, output_dim=1, rnn_type="GRU"))


def ckpt_path(tag):
    return os.path.join(MODEL_DIR, tag)

## The contingency task

`ContingencyDataset` (registered as `"contingency"`) simulates the Pavlovian sessions of the experiment. A session is a stream of trials, and one trial is

$$\underbrace{\text{nothing}}_{\text{ITI}} \;\longrightarrow\; \text{odor onset} \;\longrightarrow\; \underbrace{\text{nothing}}_{\text{ISI}} \;\longrightarrow\; \text{reward or omission},$$

discretized at dt = 0.5 s. The ISI (odor onset to outcome) is fixed at 8 steps (4 s); the ITI is a truncated geometric draw (p = 1/12, kept within [8, 20] steps) on top of a 20-step minimum, so it lasts 28–40 steps (14–20 s) and its end cannot be anticipated from timing alone. This unpredictability is the point of the experiment: how the model represents the eventless ITI decides whether it can explain degradation at all.

The model observes four channels: one-hot odor A, B, C and a reward channel (which is also the TD target). Trial types repeat in proportions 2/5 odor A, 1/5 odor B, 2/5 "blank", block-shuffled in blocks of 20 trials. The three conditions differ only in what fills the blank slots:

| condition | odor A | odor B | blank slots | mouse behavior |
|---|---|---|---|---|
| conditioning | reward, p = 0.75 | never rewarded | nothing happens | strong anticipatory licking to A |
| degradation | reward, p = 0.75 | never rewarded | **uncued** reward, p = 0.75 | licking to A collapses |
| cue-c (cued reward) | reward, p = 0.75 | never rewarded | odor C → reward, p = 0.75 | licking to A preserved |

Degradation and cue-c deliver exactly the same rewards; only their predictability differs. That is the paper's control against satiety and reward-rate accounts of contingency.

Two properties of the dataset matter for everything below. First, **episodes, not trials, are the unit of training**: twenty consecutive trials are concatenated into one stream and the RNN state is carried across the whole stream, so the ITI between two trials is experienced exactly like the eventless periods inside a trial. `sample_batch()` returns a batch of 12 episodes, zero-padded with a mask. Second, **trial labels come from the inputs, not the simulator state** (the paper's analysis convention): a degradation trial shows no cue, so input-based labeling calls it "Degradation", and an unrewarded odor-A trial looks like an odor-B trial until the outcome time has passed. The printed counts below show the resulting label distribution: in conditioning, about 300 of 400 odor-A trials are rewarded (~ 0.75), and the 400 blank slots of one condition become 400 degradation or cue-C trials in the others.

Training data jitter the reward time by one step in either direction (the paper's way of adding internal timing uncertainty); test and display data are jitter-free. The plot shows one example trial per cue/outcome for each condition; consecutive examples are laid out along the time axis.

In [ ]:
# One example trial per cue/outcome for each condition
CH_STYLE = [("cue A", "tab:green"), ("cue B", "tab:gray"),
            ("cue C", "tab:blue"), ("reward", "tab:red")]
fig, axes = plt.subplots(3, 1, figsize=(7, 5.5))
for ax, mode in zip(axes, MODES):
    ds = make_test_dataset(mode)
    shown, t_off = set(), 0
    for trial in ds.trials:
        key = (trial["cue"], trial["rewarded"])
        if key in shown:
            continue
        shown.add(key)
        for ch, (name, color) in enumerate(CH_STYLE):
            ts = np.where(trial["X"][:, ch] > 0)[0] + t_off
            ax.eventplot([ts], lineoffsets=ch, linelengths=0.8, colors=color)
        t_off += trial["length"] + 5
        if len(shown) >= 5:
            break
    ax.set_yticks(range(4), [name for name, _ in CH_STYLE])
    ax.set_title(f'condition = "{mode}"   reward probs [A, B, C] = '
                 f'{ds.rew_probs}   cue C shown: {ds.cue_shown[2]}',
                 fontsize=10)
    ax.set_xlim(-2, t_off)
axes[-1].set_xlabel("time (steps of 0.5 s); consecutive example trials offset along x")
fig.suptitle("The contingency task: example trials per condition")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "task_demo.png"), dpi=150)
plt.show()

# Trial-type statistics (labeled from the *inputs*, so hidden-cue trials
# appear as Degradation / Blank: the analysis convention of the paper)
from collections import Counter
for mode in MODES:
    ds = make_test_dataset(mode)
    print(f"{mode:13s}", dict(Counter(label_trial_type(t["X"]) for t in ds.trials)))

## The TD objective and the value-RNN

The value-RNN is a *critic-only* agent: there is no policy, because the task is Pavlovian and the animal's only job is to predict what comes next. The model is a plain GRU whose hidden state $\mathbf{z}_t = f_\phi(\mathbf{o}_t, \mathbf{z}_{t-1})$ plays the role of the state representation, with a scalar linear readout of value,

$$V_t = \mathbf{w}^\top \mathbf{z}_t + w_0.$$

All parameters (the GRU's $\phi$ and the readout $\mathbf{w}, w_0$) are trained end-to-end with the semi-gradient TD(0) loss

$$\delta_t = r_{t+1} + \gamma\, V_{t+1}\big|_{\text{stop-grad}} - V_t, \qquad \mathcal{L} = \mathbb{E}_t\big[\delta_t^2\big],$$

implemented as `TDObjective` (registered as `"td"`). Three details of the loss are worth knowing:

* *Semi-gradient*: the bootstrap target $r_{t+1} + \gamma V_{t+1}$ is treated as a constant (`.detach()`), exactly as in the paper; gradients flow only through $V_t$.
* *Reward indexing*: $r_{t+1}$ is the reward channel of the dataset's target stream, so the TD error at the step *before* an observed reward is what corresponds to the dopamine response at that reward.
* *Masking*: a TD error is counted only when both $t$ and $t+1$ are valid steps, so padded episode tails never leak into the bootstrap.

The discount factor is $\gamma = 0.83$ per 0.5 s step, a discount rate of 0.67 per second, chosen by the paper to match the $\gamma = 0.925$ of its 0.2 s tabular TD simulations. Because `TDObjective` is an ordinary objective, the generic `Trainer` drives the optimization (Adam, learning rate $10^{-3}$, batches of 12 episodes), and `keep_best=True` restores the lowest-loss weights at the end, matching the reference's argmin-over-epochs checkpoint selection.

The cell below sanity-checks the objective on one batch: an untrained network already has a small loss (rewards are sparse, so predicting near zero is not a bad start) and a mean predicted value near zero.

In [ ]:
demo_model = new_model(HIDDEN_MAIN)
demo_obj = TDObjective(gamma=GAMMA)
demo_ds = make_dataset("conditioning", seed=0)
loss0, logs = demo_obj.compute_loss(demo_model, demo_ds.sample_batch())
print(f"untrained TD loss: {loss0.item():.4f}  logs: {logs}")

## Two-phase training: conditioning, then degradation or cue-c

The networks go through the same protocol as the mice (Fig. 6b): every network is first trained to asymptote on *conditioning*, and the main models are then *retrained* on either degradation or cue-c. This transfer design is what lets the state-space comparison below attribute differences between conditions to the second-phase experience alone, since all networks of one seed share their conditioning pretraining.

One conditioning model per (hidden size, seed) is trained, then transferred to both phase-2 conditions:

| phase | modes | hidden units | seeds | used in |
|---|---|---|---|---|
| conditioning | conditioning | 5, 10, 20, 50 | 2 | all figures |
| transfer | degradation, cue-c | 5, 10, 20, 50 | 2 | Fig. 6c/d/e/g (50 units) and Fig. 6f (full sweep) |

Paper hyperparameters kept: Adam at learning rate $10^{-3}$, batches of 12 episodes of 20 trials, jittered reward timing during training. Reduced for the tutorial: 2,000-trial sessions (paper: 10,000), 3,000 + 2,000 gradient steps for the two phases (paper: 300 epochs, ~12,500 steps), 2 seeds (paper: 25). Everything is load-first: existing checkpoints under `models/19/` are reused; delete them to retrain (a full reduced run takes roughly one hour on CPU).

In [ ]:
def train_one(model, dataset, steps, tag, seed=0):
    args = TrainingArguments(max_steps=steps, learning_rate=LR,
                             batch_size=BATCH_SIZE, keep_best=True,
                             device="cpu", seed=seed, log_every=100,
                             output_dir=ckpt_path(tag))
    trainer = Trainer(model, dataset, TDObjective(gamma=GAMMA), args)
    trainer.train()
    model.save_pretrained(ckpt_path(tag))  # keep_best weights already restored
    return trainer


def train_conditioning_then(transfer_mode, hidden, seed):
    """Load or train: conditioning phase, then transfer to degradation/cue-c."""
    cond_tag = f"cond_h{hidden}_s{seed}"
    xfer_tag = f"{transfer_mode}_h{hidden}_s{seed}"
    if os.path.exists(os.path.join(ckpt_path(xfer_tag), "config.json")):
        return AutoModel.from_pretrained(ckpt_path(xfer_tag))
    if os.path.exists(os.path.join(ckpt_path(cond_tag), "config.json")):
        model = AutoModel.from_pretrained(ckpt_path(cond_tag))
    else:
        torch.manual_seed(seed); np.random.seed(seed)
        model = new_model(hidden)
        train_one(model, make_dataset("conditioning", seed),
                  STEPS_CONDITIONING, cond_tag, seed)
    train_one(model, make_dataset(transfer_mode, seed + 100),
              STEPS_TRANSFER, xfer_tag, seed)
    return model


TRAIN = True   # set False to only load existing checkpoints
if TRAIN:
    for seed in SEEDS:
        for hidden in HIDDEN_SWEEP:
            for mode in ("degradation", "cue-c"):
                train_conditioning_then(mode, hidden, seed)
                print(f"done: {mode} h{hidden} s{seed}")

The training curve (mean squared TD error per logged step) drops quickly at first, as the readout learns the average reward rate of the session, and then keeps decaying slowly while the recurrent dynamics refine the cue and ITI representations needed for precise timing. The mean predicted value settles at the average discounted reward rate of the conditioning session.

In [ ]:
# Training curve of one conditioning run (logged by the generic Trainer)
import json
hist_path = os.path.join(ckpt_path(f"cond_h{HIDDEN_MAIN}_s{SEEDS[0]}"),
                         "history.json")
hist = json.load(open(hist_path))
steps = [h["step"] for h in hist]
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(steps, [h["loss"] for h in hist], color="tab:blue")
ax.set_xlabel("gradient step")
ax.set_ylabel("TD loss (mean $\delta^2$)", color="tab:blue")
ax.tick_params(axis="y", labelcolor="tab:blue")
if "mean_value" in hist[0]:
    ax2 = ax.twinx()
    ax2.plot(steps, [h["mean_value"] for h in hist], color="tab:orange",
             alpha=0.8)
    ax2.set_ylabel("mean predicted value $V$", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")
ax.set_title(f"conditioning phase (h = {HIDDEN_MAIN}, seed {SEEDS[0]})")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "training_curve.png"), dpi=150)
plt.show()

## Asymptotic RPE predictions per condition (Fig. 6c)

The paper's central comparison: the TD error at each task event, read out from a trained network, should match the dopamine response recorded at that event in the corresponding condition. Following the paper's figure code, we draw **one panel per event** with the three conditions on the x-axis, bars showing the mean over simulations with s.d. error bars. Each trained model is probed on fresh jitter-free data of *its own* condition (the conditioning model on conditioning, and so on), trials are labeled from the inputs, and the RPE is averaged at six events: odor A onset, the rewarded outcome of odor A ("Predicted Reward"), the omitted outcome ("Omission"), odor B onset, odor C onset, and the uncued background reward of degradation ("Unpredicted Reward"). Events that never occur in a condition (odor C outside cue-c, uncued rewards outside degradation) have no bar, as in the paper.

Two analysis conventions of the paper are built into `rl.probe` and matter for reading the bars:

* *Cue-aligned time.* Each trial is re-aligned so its first event (cue onset, or the hidden-cue time for degradation trials) sits at $t = 0$; degradation trials are shifted by ISI + 1 so that the *unpredicted* background reward lands one step after where a predicted reward would have been, which is how the paper overlays conditions.
* *RPE indexed by transition start.* The RPE of the transition $t \to t+1$ is plotted at $t$, so the odor RPE appears at $-1$ and the cued-reward RPE at ISI $-1$ relative to outcome time.

The expected pattern (Fig. 6c of the paper, and the mouse dopamine data of its Fig. 2): the odor A RPE is large in conditioning, **collapses in degradation**, and is **restored in cue-c**; the RPE at the odor-A reward is positive (the reward is only 75% predictable, so a sizeable surprise remains) and nearly identical across conditions; the omission RPE is negative; and the uncued background reward in degradation evokes a large positive RPE. The paper pools 25 simulations per condition; we have 2 seeds, so treat the error bars as indicative only. The numeric values are printed under the figure.

In [ ]:
# The six events of paper Fig. 6c: (trial_type, "cue"/"reward", panel title)
EVENTS = [("Cue A Rewarded", "cue", "Cue A"),
          ("Cue A Rewarded", "reward", "Predicted Reward"),
          ("Cue A Unrewarded", "reward", "Omission"),
          ("Cue B", "cue", "Cue B"),
          ("Cue C Rewarded", "cue", "Cue C"),
          ("Degradation", "reward", "Unpredicted Reward")]


def event_rpes(mode, seed, n_episodes=10):
    """Mean RPE at each event for one trained model on its own condition."""
    tag = f"{MODEL_FOR_MODE[mode]}_h{HIDDEN_MAIN}_s{seed}"
    model = AutoModel.from_pretrained(ckpt_path(tag))
    records = probe_value_model(model, make_test_dataset(mode),
                                n_episodes=n_episodes, gamma=GAMMA)
    trials = [t for r in records for t in split_episode_trials(r)]
    return mean_rpe_at_events(trials, isi=ISI)


# rpe[seed][mode][event_title]
rpe = {s: {m: {} for m in MODES} for s in SEEDS}
for seed in SEEDS:
    for mode in MODES:
        d = event_rpes(mode, seed)
        for tt, kind, title in EVENTS:
            if tt in d and np.isfinite(d[tt][kind]):
                rpe[seed][mode][title] = d[tt][kind]

fig, axes = plt.subplots(2, 3, figsize=(7, 4), sharex=True)
x = np.arange(len(MODES))
for ax, (tt, kind, title) in zip(axes.ravel(), EVENTS):
    vals = [[rpe[s][m][title] for s in SEEDS if title in rpe[s][m]]
            for m in MODES]
    means = [np.mean(v) if v else np.nan for v in vals]
    sds = [np.std(v, ddof=1) if len(v) > 1 else 0.0 for v in vals]
    present = np.isfinite(means)
    ax.bar(x[present], np.array(means)[present],
           yerr=np.array(sds)[present], capsize=4, width=0.6,
           color=[COND_COLOR[m] for m, p in zip(MODES, present) if p])
    ax.axhline(0, color="k", lw=0.6)
    ax.set_title(title, fontsize=11)
    ax.set_xticks(x, [COND_LABEL[m] for m in MODES], fontsize=9)
for ax in axes[:, 0]:
    ax.set_ylabel("RPE (TD error)")
fig.suptitle("Fig. 6c: asymptotic value-RNN RPE predictions per condition "
             f"(mean +/- s.d. over n = {len(SEEDS)} simulations)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig6c_rpe_bars.png"), dpi=150)
plt.show()

print(f"{'event':<20s}" + "".join(f"{COND_LABEL[m]:>14s}" for m in MODES))
for tt, kind, title in EVENTS:
    row = f"{title:<20s}"
    for m in MODES:
        vals = [rpe[s][m][title] for s in SEEDS if title in rpe[s][m]]
        row += f"{np.mean(vals):>14.3f}" if vals else f"{'--':>14s}"
    print(row)

The bars recover the paper's signature. The odor A RPE, large in the conditioning model (~0.16), collapses to nearly zero in the degradation model and returns in the cue-c model (~0.09). The reward after odor A evokes the same RPE in all three conditions (~0.35; the residual surprise of a 75%-predicted reward, matching the dopamine data of the paper's Extended Data Fig. 6), omissions are negative (~−0.19), and the uncued background reward is the single largest event the degradation model ever sees (~0.7). Note also odor B: its RPE, near zero in conditioning, turns negative in degradation, because odor B resets the network from the valuable late-ITI period back to the post-outcome waiting period, again as in the paper's Fig. 5. Because odor A's own contingency is identical in all three conditions, the difference in its RPE can only come from the context the network has built around it, which is precisely the paper's argument.

## Example value and TD-error traces (Fig. 6d)

Why does the odor A RPE collapse in degradation? TD errors are *differences*: at odor onset, $\delta = \gamma V(\text{ISI}) - V(\text{pre-cue})$. The degradation manipulation could lower $\delta$ in two ways: by lowering the ISI value (odor A predicts less than before) or by raising the pre-cue value (the ITI itself has become valuable, because uncued rewards arrive there). The paper resolves this by plotting the value trace itself: the ISI value is essentially unchanged across conditions, while the **ITI value rises** in degradation. Waiting has become valuable, so the transition into the odor-A ISI is no longer a positive surprise.

As in the paper's figure code, we take one example simulation (one seed) and plot the trial-averaged value and TD error as a grid: one row per trial type, value on the left and TD error on the right, with one curve per condition in each panel (each condition's curve comes from its own trained network probed on its own condition). Time is cue-aligned with the ITI folded onto the end of the previous trial (`shift_iti`, so exactly four pre-cue steps remain), the dotted line marks cue onset and the dashed line the outcome, and the axis window (−5 to 20 steps) follows the paper. The cue-evoked RPE sits one step before cue onset, and the outcome RPE one step before the dashed line, because RPEs are indexed by transition start.

In [ ]:
ROW_TYPES = ["Cue A Rewarded", "Cue A Unrewarded", "Cue B",
             "Cue C Rewarded", "Cue C Unrewarded", "Degradation"]
ROW_LABEL = {"Cue A Rewarded": "odor A, rewarded",
             "Cue A Unrewarded": "odor A, unrewarded",
             "Cue B": "odor B",
             "Cue C Rewarded": "odor C, rewarded",
             "Cue C Unrewarded": "odor C, unrewarded",
             "Degradation": "uncued reward"}


def aligned_curves(mode, seed, n_episodes=10):
    """Trial-averaged V and RPE per trial type for one model on its condition."""
    tag = f"{MODEL_FOR_MODE[mode]}_h{HIDDEN_MAIN}_s{seed}"
    model = AutoModel.from_pretrained(ckpt_path(tag))
    records = probe_value_model(model, make_test_dataset(mode),
                                n_episodes=n_episodes, gamma=GAMMA)
    trials = [t for r in records for t in split_episode_trials(r)]
    return {key: average_by_aligned_time(trials, key=key)
            for key in ("V", "rpe")}


seed = SEEDS[0]
curves = {m: aligned_curves(m, seed) for m in MODES}

fig, axes = plt.subplots(len(ROW_TYPES), 2, figsize=(6, 8), sharex=True)
for row, tt in enumerate(ROW_TYPES):
    for col, key in enumerate(("V", "rpe")):
        ax = axes[row, col]
        for mode in MODES:
            if tt not in curves[mode][key]:
                continue
            t, m, sem, n = curves[mode][key][tt]
            ax.plot(t, m, color=COND_COLOR[mode], lw=1.4,
                    label=f"{COND_LABEL[mode]}"
                          if row == 0 and col == 0 else None)
            ax.fill_between(t, m - sem, m + sem, color=COND_COLOR[mode],
                            alpha=0.15, lw=0)
        ax.axvline(0, color="k", ls=":", lw=0.8)
        ax.axvline(ISI, color="k", ls="--", lw=0.8)
        ax.axhline(0, color="k", lw=0.4, alpha=0.4)
        ax.set_xlim(-5, 20)
        if col == 0:
            ax.set_ylabel(ROW_LABEL[tt], fontsize=9)
axes[0, 0].set_title("value $V_t$")
axes[0, 1].set_title("TD error $\delta_t$")
axes[0, 0].legend(fontsize=8, loc="upper right", borderaxespad=0)
for ax in axes[-1]:
    ax.set_xlabel(f"aligned time from cue onset")
fig.suptitle(f"trial-averaged value and TD error ")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig6d_value_rpe_traces.png"), dpi=150)
plt.show()

Read the first row (odor A, rewarded) just *before* the dotted line: the pre-cue value of the degradation network sits visibly higher than that of the conditioning network, while the value reached inside the ISI is similar in all three conditions. The odor A RPE (the bump just left of the dotted line in the right column) shrinks accordingly. The bottom row shows the manipulation itself: an uncued reward produces a large positive RPE at outcome time and immediately resets the value trace — exactly the event that teaches the degradation network that the ITI is worth waiting in. The odor B row shows the mirror image: odor B onset *drops* the value in degradation (it resets the network out of the valuable late-ITI state), which is the negative RPE seen in the bar plot above.

## CCA-aligned state-space trajectories (Fig. 6e/g)

The value-RNN was never told about the Belief-State model's states. Do its hidden dynamics nonetheless organize the task in the same way? Comparing state spaces *across conditions* needs a common coordinate system, because the three trained networks have different hidden bases. The paper's solution (its Extended Data Fig. 7 pipeline): take one simulation seed — three networks that share their conditioning pretraining and differ only in phase-2 experience — probe each network on data of its own condition, z-score and PCA-reduce each condition's hidden activity separately (keeping 95% of variance), and align the three PCA spaces with multi-view **canonical correlation analysis** (`fit_cca` in `neuralrnn.analysis`). CCA finds, for each view, a linear projection that is maximally correlated with the corresponding projections of the other views, so the three networks' trajectories can be drawn in one shared 3-D space (the first three canonical variates).

The first figure below is Fig. 6e: trial-averaged trajectories per condition, colored by trial type, with cue onset marked by a circle and the outcome by a star (the paper's markers). Three patterns of the paper are worth looking for: the **odor A ISI trajectory overlaps across conditions** (the cue-evoked dynamics are shared, because the structure of an odor-A trial never changes); in the cue-c network the **odor C trajectory tracks the odor A trajectory** (the network generalizes between the two reward-predicting cues); and the degradation network's **odor B and uncued-reward trajectories are elongated**, reflecting the Wait-state-like ITI representation.

In [ ]:
from matplotlib.lines import Line2D
from neuralrnn.analysis import fit_cca

PLOT_TYPES = ["Cue A Rewarded", "Cue A Unrewarded", "Cue B",
              "Cue C Rewarded", "Cue C Unrewarded", "Degradation", "Blank"]
TYPE_STYLE = {"Cue A Rewarded": ("tab:green", "odor A, rewarded"),
              "Cue A Unrewarded": ("tab:olive", "odor A, unrewarded"),
              "Cue B": ("tab:gray", "odor B"),
              "Cue C Rewarded": ("tab:blue", "odor C, rewarded"),
              "Cue C Unrewarded": ("tab:cyan", "odor C, unrewarded"),
              "Degradation": ("tab:red", "uncued reward"),
              "Blank": ("k", "blank (pure ITI)")}


def mean_trajectory(trials, trial_type, pre_cue_steps=4, t_max=45):
    """Type-averaged hidden trajectory on the shift_iti folded time base."""
    bucket = {}
    for trial in trials:
        if trial["trial_type"] != trial_type:
            continue
        aligned = shift_iti(trial["aligned_time"], pre_cue_steps)
        for a, z in zip(aligned, trial["Z"]):
            if -pre_cue_steps <= a < t_max:
                bucket.setdefault(int(a), []).append(z)
    if not bucket:
        return None, None
    times = np.array(sorted(bucket))
    traj = np.stack([np.mean(bucket[t], axis=0) for t in times])
    return times, traj


def plot_condition(ax, space, view, trials, iti_only=False):
    """Draw one condition's trial-type trajectories in the shared CC space."""
    for tt in PLOT_TYPES:
        times, traj = mean_trajectory(trials, tt)
        if traj is None:
            continue
        V = space.transform(traj, view)
        if iti_only:  # keep pre-cue (-4..-1) and post-outcome (> ISI) steps
            keep = (times < 0) | (times > ISI)
            times, V = times[keep], V[keep]
            gap = np.where(np.diff(times) > 1)[0] + 1  # break at the ISI
        else:
            gap = []
        color, label = TYPE_STYLE[tt]
        for seg in np.split(np.arange(len(times)), gap):
            ax.plot(
                V[seg, 0], 
                V[seg, 1], 
                # V[seg, 2], 
                color=color, lw=1.5, alpha=0.6 if tt == "Blank" else 0.95)
        if not iti_only:  # mark cue onset (t = 0) and outcome (t = ISI)
            for t0, mk in ((0, "o"), (ISI, "*")):
                if t0 in times:
                    p = V[list(times).index(t0)]
                    ax.scatter(*p[:2], marker=mk, color=color,
                               s=45 if mk == "*" else 25,
                               edgecolor="k", linewidth=0.4, zorder=5)
    ax.set_xlabel("CC1", labelpad=-8)
    ax.set_ylabel("CC2", labelpad=-8)
    # ax.set_zlabel("CC3", labelpad=-8)
    ax.tick_params(pad=-4, labelsize=7)


seed = SEEDS[0]
# one trained network per condition (same seed, shared conditioning
# pretraining), each probed on its own condition
per_mode = {}
for mode in MODES:
    model = AutoModel.from_pretrained(
        ckpt_path(f"{MODEL_FOR_MODE[mode]}_h{HIDDEN_MAIN}_s{seed}"))
    records = probe_value_model(model, make_test_dataset(mode),
                                n_episodes=10, gamma=GAMMA)
    per_mode[mode] = [t for r in records for t in split_episode_trials(r)]
acts = [np.concatenate([t["Z"] for t in per_mode[m]], axis=0) for m in MODES]
space = fit_cca(acts, var_threshold=0.95, n_components=3)

handles = [Line2D([], [], color=c, lw=2, label=l)
           for c, l in (TYPE_STYLE[t] for t in PLOT_TYPES)]
fig = plt.figure(figsize=(8, 3))
for view, mode in enumerate(MODES):
    ax = fig.add_subplot(1, 3, view + 1)
    plot_condition(ax, space, view, per_mode[mode])
    ax.set_title(f"{COND_LABEL[mode]} network", fontsize=11)
    ax.axis('off')
fig.legend(handles=handles, fontsize=8, loc="center right",
           bbox_to_anchor=(1.0, 0.5))
fig.suptitle(f"hidden-state trajectories in the CCA-aligned space "
             f"(circle = cue onset, star = outcome)")
fig.tight_layout(rect=(0, 0, 0.87, 1))
fig.savefig(os.path.join(FIG_DIR, "fig6e_cca_trajectories.png"), dpi=150)
plt.show()

In the cue-c network the blue (odor C) trajectory rides on top of the green (odor A) one: having learned that both cues predict the same outcome, the network routes them through the same ISI dynamics. The odor A trajectories themselves are similar across the three networks, as in the paper.

The second figure is Fig. 6g: the same trajectories restricted to the ITI portion — the four pre-cue steps plus the post-outcome return path. Here the degradation network stands out: after an outcome its state takes a long, separate loop back towards the ITI region, regardless of which trial type preceded it, while the conditioning and cue-c networks collapse quickly onto a tight ITI bundle. This heterogeneous, slow-returning ITI is the state-space signature of the Belief-State model's Wait state, which the degradation network had to invent because uncued rewards make "how long have I been waiting" behaviorally relevant.

In [ ]:
fig = plt.figure(figsize=(8, 3))
for view, mode in enumerate(MODES):
    ax = fig.add_subplot(1, 3, view + 1)
    plot_condition(ax, space, view, per_mode[mode], iti_only=True)
    ax.set_title(f"{COND_LABEL[mode]} network", fontsize=11)
    ax.axis('off')
fig.legend(handles=handles, fontsize=8, loc="center right",
           bbox_to_anchor=(1.0, 0.5))
fig.suptitle(f"hidden-state trajectories in the CCA-aligned space "
             f"(ITI period only)")
fig.tight_layout(rect=(0, 0, 0.87, 1))
fig.savefig(os.path.join(FIG_DIR, "fig6g_cca_iti.png"), dpi=150)
plt.show()

## Decoding the hand-crafted beliefs (Fig. 6f)

The strongest version of the paper's claim: the value-RNN's state space resembles the Belief-State model's, and more so the larger the network. The Belief-State model is a hidden Markov model over 10 latent task states: 8 ISI substates (one per 0.5 s step from odor onset to outcome), a **Wait** state (post-outcome ITI) and a **Pre** state (late ITI, when the next trial has become likely). Its transition matrix $T_{ij} = P(s_{t+1} = j \mid s_t = i)$ and observation matrix $O_{ij,e} = P(\text{event } e \mid s_t \to s_{t+1})$ over the five events (null, reward, odor A, B, C) are estimated empirically from a simulated session with latent-state labels, *separately for each condition* (`simulate_belief_experiment`, `estimate_transition`, `estimate_observation`). Given an observed event stream, the belief trajectory is the HMM forward filter

$$b_t \;=\; \mathrm{normalize}\Big( b_{t-1}\, \big(T \odot O_{:,:,\,e_t}\big) \Big),$$

a 10-dimensional probability vector at every step. To ask whether the RNN encodes these beliefs, a linear map from hidden activity to beliefs is fit by least squares, $B \approx [Z, \mathbf{1}]\,W$, and scored with $R^2 = 1 - \mathrm{var}(B - \hat B)\,/\,\mathrm{var}(B)$, computed over all 10 beliefs together and for the Wait state alone (`belief_regression_r2`).

Following the paper's analysis code, each trained network is probed on its own condition, the probed episodes are concatenated into one event stream, the beliefs are filtered with *that condition's* matrices, and one regression is fit over the whole stream. The sweep then repeats this for every hidden size (5–50 units), every condition and both seeds. The paper's result (its Fig. 6f): belief R² **grows with the number of hidden units**, and the improvement concentrates in the Wait state, the ITI representation whose heterogeneity separates degradation from the other conditions.

### Belief-state reference model (notebook utilities)

The functions below are the Belief-State reference model of the paper, ported from the authors' MATLAB code (`simulate_experiment_rnn.m`, `simulate_trial.m`, `td_belief.m`, `belief_r2.m`). They are specific to this Pavlovian paradigm, so they live in this notebook rather than in the package. The port keeps the reference's conventions exactly (1-based event codes; the belief relabeling that collapses the post-outcome ISI of unrewarded trials into the Wait state; the minimum-norm least-squares decoder), and was verified to reproduce the earlier full-length port bit-for-bit, so the belief R² values are directly comparable to the paper's.

In [ ]:
# --- Belief-State reference model (port of the authors' MATLAB code) ---
# Latent states before relabeling: ISI substates 0..8 (cue at 0, outcome at 8;
# dt = 0.5 s) and the post-outcome wait chain 8..20 (20 = late-ITI "Pre").
# The belief relabeling of simulate_trial.m collapses every state in (7, 20)
# into 19 ("Wait"), leaving 10 dense states: ISI 0..7, Wait, Pre.
# Events: 1 = null, 2 = reward, 3/4/5 = odor A/B/C.
EV_NULL, EV_REWARD = 1, 2
ISI_STATES = np.arange(9)
WAIT_CHAIN = np.arange(8, 21)
PRE, WAIT = 20, 19

TRIAL_DIST = {
    "conditioning": ["CSp", "CSp", "CSm", "blank", "blank"],
    "degradation": ["CSp", "CSp", "CSm", "degrade", "degrade"],
    "cue-c": ["CSp", "CSp", "CSm", "CSp2", "CSp2"],
    "extinction": ["CSpExtinct", "CSpExtinct", "CSm", "blank", "blank"],
}


def _belief_trial(ttype, rng, p_reward=0.75):
    """Latent states and events of one trial (trial + trailing ITI)."""
    iti = rng.exponential(6.0)     # trailing ITI: exp(6 s) kept in [4, 10] s
    while not 4.0 <= iti <= 10.0:
        iti = rng.exponential(6.0)
    tail = np.full(int(np.ceil((iti - 4.0) / 0.5)), PRE)
    p_rand = rng.rand()  # drawn for every trial, as simulate_trial.m
    rewarded = ttype in ("CSp", "CSp2", "degrade") and p_rand < p_reward

    isi_ev = np.ones(9, int)
    isi_ev[0], isi_ev[-1] = -1, 0  # placeholders: cue event, outcome event
    null_tail = np.ones(len(WAIT_CHAIN) + len(tail), int)

    if ttype in ("CSp", "CSp2", "CSpExtinct", "CSm"):  # cued trials
        cue_ev = {"CSp": 3, "CSm": 4, "CSp2": 5, "CSpExtinct": 3}[ttype]
        events = np.concatenate([[1, 1], isi_ev, null_tail])
        events[events == -1] = cue_ev
        events[events == 0] = EV_REWARD if rewarded else EV_NULL
        if ttype == "CSm":  # odor B: cue, but the ISI is spent in Pre
            states = np.concatenate([[PRE, PRE], WAIT_CHAIN, tail,
                                     np.full(9, PRE)])
        else:
            states = np.concatenate([[PRE, PRE], ISI_STATES, WAIT_CHAIN, tail])
    elif ttype == "blank":
        states = np.full(2 + 9 + len(WAIT_CHAIN) + len(tail), PRE)
        events = np.ones_like(states)
    elif ttype == "degrade":  # hidden cue: no ISI states, uncued reward
        states = np.concatenate([[PRE, PRE], np.full(9, PRE), WAIT_CHAIN, tail])
        events = np.concatenate([[1, 1], isi_ev, null_tail])
        events[events == -1] = EV_NULL
        events[events == 0] = EV_REWARD if rewarded else EV_NULL
        events = np.concatenate([[EV_NULL], events[:-1]])  # reward at arrival

    # belief relabeling: unrewarded cued trials sit in Wait during the ISI,
    # and the whole post-outcome chain collapses into Wait
    if ttype in ("CSp", "CSp2") and not rewarded:
        states = np.minimum(states + 7, PRE)
    states[(states > 7) & (states < PRE)] = WAIT
    return states, events


def simulate_belief_experiment(n_trials, condition, seed=0, p_reward=0.75):
    """(states, events) stream of one condition with latent-state labels."""
    rng = np.random.RandomState(seed)
    trials = [_belief_trial(t, rng, p_reward)
              for t in rng.choice(TRIAL_DIST[condition], size=n_trials)]
    states = np.concatenate([t[0] for t in trials])
    events = np.concatenate([t[1] for t in trials])
    remap = {u: i for i, u in enumerate(np.unique(states))}  # dense 0..S-1
    return np.array([remap[s] for s in states]), events


def estimate_transition(states):
    """Empirical T[i, j] = P(s_{t+1} = j | s_t = i)."""
    S = int(states.max()) + 1
    T = np.zeros((S, S))
    np.add.at(T, (states[:-1], states[1:]), 1)
    out = T.sum(axis=1, keepdims=True)
    return np.divide(T, out, out=np.zeros_like(T), where=out > 0)


def estimate_observation(events, states, n_events=5):
    """Empirical O[i, j, e] = P(event e at arrival | transition i -> j)."""
    S = int(states.max()) + 1
    O = np.zeros((S, S, n_events))
    np.add.at(O, (states[:-1], states[1:], events[1:] - 1), 1)
    tot = O.sum(axis=2, keepdims=True)
    return np.divide(O, tot, out=np.zeros_like(O), where=tot > 0)


def belief_filter(events, T, O):
    """HMM forward filter; b_t stored BEFORE observing e_t (td_belief.m)."""
    S = T.shape[0]
    b = np.ones(S) / S
    out = np.zeros((len(events), S))
    for t, e in enumerate(events):
        out[t] = b
        b = b @ (T * O[:, :, e - 1])
        s = b.sum()
        b = b / s if np.isfinite(s) and s > 0 else np.eye(S)[-1]
    return out


def belief_regression_r2(Z, B):
    """Least-squares decode of beliefs B from hidden activity Z (+ intercept).

    Returns (total R^2, per-belief R^2): the total centers residual and data
    columns (belief_r2.m); per-belief is 1 - SSR/SST per column (r2_matrix.m).
    """
    Zb = np.hstack([Z, np.ones((len(Z), 1))])
    W = np.linalg.lstsq(Zb, B, rcond=None)[0]
    resid = B - Zb @ W
    r2_total = 1 - np.sum((resid - resid.mean(0)) ** 2) / \
        np.sum((B - B.mean(0)) ** 2)
    sst = np.sum((B - B.mean(0)) ** 2, axis=0)
    r2_per = 1 - np.sum(resid ** 2, axis=0) / np.maximum(sst, 1e-12)
    return float(r2_total), r2_per

In [ ]:
WAIT_COL = 8   # Wait-state belief column (dense states: ISI 0..7, Wait, Pre)


def events_from_X(X):
    """Input channels -> belief-model events (1=null, 2=reward, 3=A, 4=B, 5=C)."""
    e = np.ones(X.shape[0], dtype=int)
    e[X[:, 3] > 0] = 2
    e[X[:, 0] > 0] = 3
    e[X[:, 1] > 0] = 4
    e[X[:, 2] > 0] = 5
    return e


def belief_r2_for_model(tag, mode, T, O, n_episodes=10):
    """Belief R^2 of one trained model on its own condition's event stream."""
    model = AutoModel.from_pretrained(ckpt_path(tag))
    records = probe_value_model(model, make_test_dataset(mode),
                                n_episodes=n_episodes, gamma=GAMMA)
    X = np.concatenate([r["X"] for r in records])  # one continuous stream,
    Z = np.concatenate([r["Z"] for r in records])  # as in the paper's analysis
    B = belief_filter(events_from_X(X), T, O)
    r2_total, r2_per = belief_regression_r2(Z, B)
    return r2_total, float(r2_per[WAIT_COL])


# Belief-State transition/observation matrices per condition (one large
# labeled simulation each; reference: nSims = 4000)
TO = {}
for mode in MODES:
    states, events = simulate_belief_experiment(4000, mode, seed=0)
    TO[mode] = (estimate_transition(states),
                estimate_observation(events, states))
    print(f"{mode}: {states.max() + 1} belief states")

# r2[mode][hidden] = list of (total, wait) over seeds
r2 = {m: {h: [] for h in HIDDEN_SWEEP} for m in MODES}
for mode in MODES:
    T, O = TO[mode]
    for hidden in HIDDEN_SWEEP:
        for seed in SEEDS:
            tag = f"{MODEL_FOR_MODE[mode]}_h{hidden}_s{seed}"
            r2[mode][hidden].append(belief_r2_for_model(tag, mode, T, O))
            print(f"{mode} h={hidden:3d} s={seed}: "
                  f"total R2={r2[mode][hidden][-1][0]:.3f}  "
                  f"wait R2={r2[mode][hidden][-1][1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, col, title in ((axes[0], 0, "all 10 beliefs"),
                       (axes[1], 1, "Wait state only")):
    for mode in MODES:
        means = [np.mean([v[col] for v in r2[mode][h]]) for h in HIDDEN_SWEEP]
        sds = [np.std([v[col] for v in r2[mode][h]], ddof=1)
               if len(r2[mode][h]) > 1 else 0.0 for h in HIDDEN_SWEEP]
        ax.errorbar(HIDDEN_SWEEP, means, yerr=sds, marker="o", ms=4,
                    capsize=3, color=COND_COLOR[mode],
                    label=f"{COND_LABEL[mode]} network")
    ax.set_xlabel("number of hidden units")
    ax.set_ylabel("belief $R^2$")
    ax.set_xticks(HIDDEN_SWEEP)
    ax.set_ylim(0, 1.05)
    ax.set_title(title, fontsize=11)
axes[0].legend(fontsize=9)
fig.suptitle(f"larger value-RNNs are more belief-like")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig6f_belief_r2.png"), dpi=150)
plt.show()

The sweep reproduces the paper's trend in all three conditions: the total belief R² climbs from 0.2–0.3 at 5 units to 0.90–0.99 at 50 units, and the Wait-state R² climbs with it (from below 0.3 to above 0.93). A 50-unit GRU trained only to predict reward has learned an almost complete linear embedding of the hand-crafted beliefs, while a 5-unit network cannot afford one. The Wait-state panel also shows the paper's cross-condition pattern: at small and intermediate sizes the degradation and cue-c networks decode the Wait state better than the conditioning network (at 10 units, 0.47–0.73 versus 0.23–0.42) — exactly the heterogeneous ITI representation that the uncued (or newly cued) rewards force these networks to build, and the same signature visible as the elongated ITI trajectories in Fig. 6g.

## Summary

* TD learning is an *objective*, not a trainer. `TDObjective` plus the generic `Trainer` turns any RNN core with a scalar readout into a value-RNN; no dedicated RL machinery is needed for critic-only paradigms (contrast with the PPO actor-critic of notebook 18).
* `ContingencyDataset` ports the paper's task: block-shuffled cues (2/5 A, 1/5 B, 2/5 blank), truncated-geometric ITIs, jittered reward timing, and episodes of 20 concatenated trials with the RNN state carried across trials.
* `rl.probe` ports the paper's analysis conventions (input-based trial labels, cue-aligned time with the degradation shift, RPE indexed by transition start, ITI folding), so value and RPE analyses transfer directly from the paper.
* The results reproduce Fig. 6 of the paper at reduced scale, with the paper's own figure layouts: the odor A RPE collapses in degradation and is restored in cue-c because the *ITI* value rises (Fig. 6c/d); the three networks of one seed share the odor-A ISI trajectory in a CCA-aligned space, but the degradation network develops a distinct, slowly returning ITI trajectory (Fig. 6e/g); and larger networks linearly decode more of the hand-crafted beliefs in every condition, especially the Wait state (Fig. 6f).

For the full-scale protocol (10,000-trial sessions, 300 epochs per phase, 25 simulations), increase `N_TRIALS`, `STEPS_*` and `SEEDS` in the setup cell.